# Phase 5A — Machine Learning Fundamentals

**Theory:** Bias-variance tradeoff, overfitting/underfitting, train/val/test split, cross-validation, evaluation metrics.

**Install:** `pip install scikit-learn`

---

In [ ]:
try:
    import sklearn

    print(f"scikit-learn version: {sklearn.__version__}")
except ImportError:
    print("Install: pip install scikit-learn")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    learning_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.datasets import make_classification, make_regression

np.random.seed(42)
sns.set_theme(style="whitegrid")

---
## 1. The ML Workflow

Every supervised ML project follows this pattern:
```
Data → Split → Preprocess → Fit → Predict → Evaluate → Tune → Deploy
```

In [ ]:
# Generate a synthetic classification dataset
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=6, n_redundant=2, random_state=42
)

print(f"Features shape: {X.shape}")
print(f"Labels shape  : {y.shape}")
print(f"Class balance : {np.bincount(y)}")

In [ ]:
# Train / Validation / Test split
# Never use test set for any decision — only final evaluation
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Training   : {X_train.shape[0]} samples")
print(f"Validation : {X_val.shape[0]} samples")
print(f"Test       : {X_test.shape[0]} samples")

# Feature scaling — ALWAYS fit on train, transform train+val+test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("\nScaling: fit on train, applied to val and test (no data leakage)")

---
## 2. Bias-Variance Tradeoff

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Generate simple 1D regression data
np.random.seed(42)
x_bv = np.sort(np.random.uniform(0, 1, 30))
y_bv = np.sin(2 * np.pi * x_bv) + np.random.normal(0, 0.3, 30)

x_dense = np.linspace(0, 1, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = [
    "Underfitting (degree=1)\nHigh Bias, Low Variance",
    "Good Fit (degree=4)",
    "Overfitting (degree=15)\nLow Bias, High Variance",
]

for ax, degree, title in zip(axes, [1, 4, 15], titles):
    pipe = Pipeline([("poly", PolynomialFeatures(degree)), ("lr", LinearRegression())])
    pipe.fit(x_bv.reshape(-1, 1), y_bv)
    y_pred_curve = pipe.predict(x_dense)

    ax.scatter(x_bv, y_bv, color="steelblue", s=40, label="Data", zorder=3)
    ax.plot(x_dense, y_pred_curve, "r-", lw=2, label=f"deg={degree}")
    ax.plot(x_dense, np.sin(2 * np.pi * x_dense), "g--", lw=1, alpha=0.5, label="True")
    ax.set_ylim(-2, 2)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Bias-Variance Tradeoff", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Cross-Validation

In [ ]:
# K-Fold cross-validation: more reliable than a single train/val split
model = LogisticRegression(random_state=42, max_iter=1000)

cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")

print("5-Fold Cross-Validation Results:")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i + 1}: {score:.4f}")
print(f"\nMean accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("(mean ± std tells you performance AND stability)")

---
## 4. Classification Metrics

In [ ]:
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_val_scaled)

print("=== Classification Report ===")
print(classification_report(y_val, y_pred))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"],
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

print("Key metrics:")
print(
    f"  Precision: {precision_score(y_val, y_pred):.3f}  (of predicted positives, how many are actually positive?)"
)
print(
    f"  Recall   : {recall_score(y_val, y_pred):.3f}  (of actual positives, how many did we catch?)"
)
print(
    f"  F1       : {f1_score(y_val, y_pred):.3f}  (harmonic mean of precision and recall)"
)

---
## 5. Regression Metrics

In [ ]:
X_reg, y_reg = make_regression(n_samples=500, n_features=5, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_tr, y_tr)
y_pred_reg = lr.predict(X_te)

rmse = mean_squared_error(y_te, y_pred_reg) ** 0.5
mae = mean_absolute_error(y_te, y_pred_reg)
r2 = r2_score(y_te, y_pred_reg)

print("=== Regression Metrics ===")
print(f"RMSE : {rmse:.3f}  (in the same units as target — penalizes large errors more)")
print(f"MAE  : {mae:.3f}   (average absolute error — more interpretable)")
print(f"R²   : {r2:.4f}  (proportion of variance explained)")

---
## Summary — sklearn API Pattern

All sklearn estimators follow the same interface:
```python
model = SomeAlgorithm(hyperparams)   # 1. Create
model.fit(X_train, y_train)          # 2. Train
y_pred = model.predict(X_test)       # 3. Predict
score  = model.score(X_test, y_test) # 4. Evaluate (R² for regression, accuracy for classification)
```

**Golden rules:**
1. Always split data BEFORE any preprocessing
2. Fit all transformers (scalers, encoders) ONLY on training data
3. Never touch the test set until final evaluation
4. Use cross-validation for model selection, not the test set